# KAT-VAD v6 — **LaGoVAD-AP pilot**: agent prompts on A2, Nexar-only, DoTA-CAP-dev zero-shot

Rules `core/docs/v6/PREREG_V6.md` (§1 Step 0, §2 build, §3 read-out). Plan `.project/plans/katvad-v6-agent-prompt.md`.
One seed (s2024). Control = **v5 R1 s2024** (VideoMAE only, no MQR, 0.8102), read from its existing protocol-B output on
Drive (same harness, same CAP-dev clips) — no v2 checkpoint is loaded or rescored. Outcome label **PILOT (n = 1)**.

| step | what | GPU? |
|---|---|---|
| 1 | DoTA-CAP-**dev** agents (569 clips): YOLOv9-C + BoT-SORT (CMC, no ReID) on every DoTA frame, agent rows at protocol-B `s1[::3]` (`v6_agents extract-dota --dev-only`) | yes |
| 2 | **Step 0** go / no-go (D0 recall, D1 load, D2 training-free ranking vs centre / largest / lowest) → stop on **NO-GO** | no |
| 3 | Nexar window sources: agents at A2's `s1[::8]` (`v6_agents extract-nexar`, sharded) | yes |
| 4 | train **AP** s2024 = A2's exact recipe + `agent.enabled=true` (`tanh(α_o)` gate, α_o = 0 ⇒ A2 at init, `L_agent` λ 0.5 K 1 β 0.01 dense negatives), 1,749 steps, J10 | yes |
| 5 | protocol B on DoTA-CAP-dev for **AP s2024** + AP candidate-logit dump | yes |
| 6 | read-out: macro / micro / strat-50, **ego vs non-ego**, paired Δ AP − R1, box-level hit vs priors → **paste `REPORTS/v6_readout.md` to Claude** | no |

**Drive.** Code `Thesis-V6/kat-vad` (upload branch `v6`). Writes `Thesis-V6/cache/{agents,tracks}/` and
`Thesis-V6/outputs/v6/`. `Thesis/`, `Thesis-V2/`, `Thesis-V5/` are read-only. DoTA-CAP-eval is sealed: not read here.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (Nexar, DoTA, MMAU/CAP), cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only INPUT DATA only (Nexar windows / census / A2-row bake, CAP alignment)
DRIVE_V5 = '/content/drive/MyDrive/Thesis-V5'     # read-only: v5 R1 protocol-B eval (the control row)
DRIVE_V6 = '/content/drive/MyDrive/Thesis-V6'     # code + everything v6 writes
os.environ['PROJECT_ROOT']       = DRIVE_V6
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V6}/outputs'

os.environ['REPO'] = f'{DRIVE_V6}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload the v6 branch there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V6_ROW_WIDTH'), 'this upload predates v6 -- re-upload kat-vad/ from branch v6'

MAX_GPU_JOBS = 2      # concurrent GPU processes (detector + VideoMAE + CLIP each); 1 on CUDA OOM
MIRROR_EVERY = 300    # seconds between cache mirrors VM -> Drive
NEXAR_SHARDS = 3      # Nexar extraction processes on one GPU

DATASET  = constants.NEXAR_DATASET
C        = constants.NEXAR_CONSTRUCTION_WINDOW
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
S_TRAIN, S_DOTA = constants.V6_NEXAR_STRIDE, constants.V6_DOTA_STRIDE      # 8, 3
SEED     = constants.V6_SEED                                               # 2024
ROW      = constants.V6_ROW_AP

V2_CACHE = Path(DRIVE_V2) / 'cache'
V2_OUT   = Path(DRIVE_V2) / 'outputs'
RAW      = constants.DATA_ROOT / DATASET / 'raw'
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
DATA     = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C}'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
CAP_RAW  = constants.DATA_ROOT / 'MMAU' / 'raw'
DCAP     = V2_OUT / 'REPORTS' / 'dota_cap'
ALIGN, FINAL = DCAP / 'dota_cap_alignment.json', DCAP / 'dota_cap_readout.json'
CAP_GROUPS = ['1-10', '11', '12-42', '43', '44-62']
BUILD    = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n4' / 'build_report.json'
A2_BAKE  = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C}' / constants.KNN_CACHE_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_A2  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
R1_EVAL  = Path(DRIVE_V5) / 'outputs' / 'v5' / 'v5_stage2' / 'eval' / 'dota' / CAP_DEV / 'R1'

CACHE6   = Path(DRIVE_V6) / 'cache'
D_AG     = {'dota': CACHE6 / constants.V6_AGENT_CACHE_DIRNAME / f'{constants.DOTA_CAP_DATASET}_s{S_DOTA}_k{constants.V6_K_MAX}',
            'train': CACHE6 / constants.V6_AGENT_CACHE_DIRNAME / f'{DATASET}_s{S_TRAIN}_k{constants.V6_K_MAX}'}
D_TR     = {'dota': CACHE6 / constants.V6_TRACK_CACHE_DIRNAME / constants.DOTA_CAP_DATASET,
            'train': CACHE6 / constants.V6_TRACK_CACHE_DIRNAME / DATASET}
ROOT     = constants.OUTPUT_ROOT / constants.V6_OUTPUT_DIRNAME
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/v6ap')
L_AG     = {k: LOCAL / 'agents' / v.name for k, v in D_AG.items()}
L_TR     = {k: LOCAL / 'tracks' / v.name for k, v in D_TR.items()}
L_REP    = LOCAL / 'reports'          # every tool writes here; mirrored to REPORTS
LOGS     = LOCAL / 'logs'
for p in (REPORTS, L_REP, LOGS, *D_AG.values(), *D_TR.values(), *L_AG.values(), *L_TR.values()):
    p.mkdir(parents=True, exist_ok=True)
assert constants.OUTPUT_ROOT == Path(DRIVE_V6) / 'outputs', constants.OUTPUT_ROOT

for name, path in (('Nexar raw', RAW), ('census', CENSUS), ('Nexar_window', DATA), ('DoTA clip s1', DOTA_S1),
                   ('CAP chunks', CAP_RAW / 'CAP-DATA_chunks'), ('CAP annotation', CAP_RAW / 'cap_text_annotations.xls'),
                   ('alignment', ALIGN), ('final ids', FINAL), ('build report', BUILD), ('A2 bake', A2_BAKE),
                   ('KNN', KNN), ('DoTA-CAP-dev A2 bake', DOTA_A2),
                   ('DoTA labels', DOTA / 'labels_s8'), ('v5 R1 eval', R1_EVAL / 'clip_scores.npz')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:20s} {path}   OK')
PLAN = json.loads(BUILD.read_text())['plan'][C]
STEPS, EPOCHS = PLAN['steps'], PLAN['epochs']                     # 1,749 = 33 x 53 (v2 §29)
print(f'AP s{SEED} | {EPOCHS} epochs = {STEPS} steps')
print('caches ->', CACHE6, '\nruns + REPORTS ->', ROOT)


In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn xlrd pandas "ultralytics==8.3.*" lap
python -c "import av, einops, faiss, sklearn, ultralytics; print('deps OK, ultralytics', ultralytics.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 1, 3-5 need one'
nproc; free -g | head -2; df -h /content | tail -1


In [ ]:
import pickle
import shutil
import subprocess
import threading
import time

import numpy as np
import torch

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
CPUS = os.cpu_count() or 8
PART = constants.CACHE_PART_SUFFIX
assert shutil.disk_usage('/content').free / 1e9 >= 60, 'need >= 60 GB free on /content (agent caches + CAP stream + staging)'


def run(cmd, log=None):
    '''One CPU / sequential command, output streamed here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def last_line(path):
    try:
        lines = [l for l in path.read_text(errors='replace').splitlines() if l.strip()]
    except FileNotFoundError:
        return '(no output yet)'
    return lines[-1][-160:] if lines else '(no output yet)'


class Job:
    '''Commands run in order in one process slot; ``after`` runs in this kernel once all succeeded.'''

    def __init__(self, name, cmds, after=None):
        self.name, self.after = name, after
        self.cmds = [[str(c) for c in cmd] for cmd in cmds]
        self.log = LOGS / f'{name}.log'


def run_parallel(jobs, max_jobs=MAX_GPU_JOBS, every=120):
    '''Run independent jobs ``max_jobs`` at a time, one GPU. A failure does not stop the others; the cell raises
    after all have ended, with each failed job's log tail.'''
    if not jobs:
        print('nothing to run')
        return
    slots = max(1, min(max_jobs, len(jobs)))
    env = dict(ENV, OMP_NUM_THREADS=str(max(1, CPUS // slots)), MKL_NUM_THREADS=str(max(1, CPUS // slots)))
    pending, running, failed, t0, shown = list(jobs), {}, [], time.time(), time.time()

    def launch(job, i):
        fh = open(job.log, 'a')
        fh.write(f'\n$ {" ".join(job.cmds[i])}\n')
        fh.flush()
        return subprocess.Popen(job.cmds[i], cwd=str(REPO), env=env, stdout=fh, stderr=subprocess.STDOUT), fh

    def stamp():
        return f'[{(time.time() - t0) / 60:6.1f} min]'

    while pending or running:
        while pending and len(running) < slots:
            job = pending.pop(0)
            job.t0 = time.time()
            running[job.name] = (job, 0, *launch(job, 0))
            print(f'{stamp()} start {job.name}  (log {job.log})')
        time.sleep(10)
        for name, (job, i, proc, fh) in list(running.items()):
            rc = proc.poll()
            if rc is None:
                continue
            fh.close()
            del running[name]
            if rc:
                failed.append(job)
                print(f'{stamp()} FAILED {name} (exit {rc})')
            elif i + 1 < len(job.cmds):
                running[name] = (job, i + 1, *launch(job, i + 1))
            else:
                try:
                    if job.after:
                        job.after()
                    print(f'{stamp()} done {name} in {(time.time() - job.t0) / 60:.1f} min')
                except Exception as exc:  # a failed publish / check is this job's failure, not the others'
                    failed.append(job)
                    print(f'{stamp()} FAILED {name} after the run: {exc!r}')
        if running and time.time() - shown > every:
            shown = time.time()
            for name, (job, *_rest) in running.items():
                print(f'           {name}: {last_line(job.log)}')
    if failed:
        for job in failed:
            tail = job.log.read_text(errors='replace').splitlines()[-30:]
            print('=' * 30, f'\n{job.name} -- paste to Claude:\n' + '\n'.join(tail))
        raise RuntimeError(f'failed: {[j.name for j in failed]}')


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10, lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file() and not p.name.endswith(PART)]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2).'''
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.partial')
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def sync_dir(src, dst, restore=False):
    '''Copy every finished file of ``src`` missing (or, for json / npz, changed) in ``dst``; in-flight writes skipped.'''
    n = 0
    for f in sorted(src.glob('*')):
        if not f.is_file() or f.name.endswith(PART) or f.name.endswith('.partial'):
            continue
        g = dst / f.name
        small = f.suffix in ('.json', '.npz', '.md')
        if g.exists() and (g.stat().st_size == f.stat().st_size) and not (small and not restore):
            continue
        if restore:
            dst.mkdir(parents=True, exist_ok=True)
            shutil.copy2(f, g)
        else:
            publish(f, g)
        n += 1
    return n


class Mirror(threading.Thread):
    '''Step 1: copy finished grid files + reports VM -> Drive every ``MIRROR_EVERY`` s, and once more at stop.'''

    def __init__(self, pairs):
        super().__init__(daemon=True)
        self.pairs, self.halt, self.copied = pairs, threading.Event(), 0

    def run(self):
        while not self.halt.wait(MIRROR_EVERY):
            self.sync()

    def sync(self):
        for src, dst in self.pairs:
            try:
                self.copied += sync_dir(src, dst)
            except OSError as exc:     # a Drive hiccup: the next round retries
                print('mirror:', exc)

    def stop(self):
        self.halt.set()
        self.join()
        self.sync()
        print(f'mirror: {self.copied} files copied to Drive')


def ckpt_step(path):
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def show(name):
    p = L_REP / f'{name}.md'
    print(p.read_text() if p.exists() else f'(no {p.name})')


sync_dir(REPORTS, L_REP, restore=True)     # earlier sessions' reports: every skip below reads them locally
print('reports restored:', sorted(p.name for p in L_REP.glob('*.json')))


In [ ]:
# DoTA annotations (boxes, ego flag): Drive copy, else the official zip
ANN = DOTA / 'raw_annotations' / 'annotations'
if not ANN.is_dir():
    ANN = LOCAL / 'dota_meta' / 'annotations'
    if not ANN.exists():
        ANN.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(f'cd {ANN.parent} && curl -sSL -o ann.zip https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip && unzip -q ann.zip && rm ann.zip',
                       shell=True, check=True)
print('annotations:', len(list(ANN.glob('*.json'))))
# YOLOv9-C weights once on VM disk; the tools get the path (--weights), never download into REPO on Drive
WEIGHTS = LOCAL / constants.V6_DETECTOR_WEIGHTS
if not WEIGHTS.exists():
    from ultralytics import YOLO
    os.chdir(LOCAL)
    YOLO(constants.V6_DETECTOR_WEIGHTS)
    os.chdir('/content')
assert WEIGHTS.exists(), WEIGHTS
ENV['YOLO_CONFIG_DIR'] = str(LOCAL / 'yolo_cfg')
for k in ('dota', 'train'):          # resume: Drive's partial caches back to VM disk; the tools skip finished videos
    n = sync_dir(D_AG[k], L_AG[k], restore=True) + sync_dir(D_TR[k], L_TR[k], restore=True)
    print(f'{k}: {n} cached files restored from Drive')


## 1. DoTA-CAP-dev agents (GPU)

CAP-DATA is streamed group by group (as v5 Stage 0); each DoTA clip is rebuilt from its aligned CAP frames, its frame
count checked against `DoTA_s1_ncc`, its aspect against 16:9. Detection + tracking on every DoTA frame (10 fps).

In [ ]:
from core.data.v2_splits import load_split

CAP_IDS = sorted(load_split(CAP_DEV))


def dota_report():
    report = {}
    for g in CAP_GROUPS:
        p = L_REP / f'extract_agents_dota_{g}.json'
        if p.exists():
            report |= json.loads(p.read_text())
    return report


def dota_done():
    return all((L_AG['dota'] / f'{v}.npy').exists() for v in CAP_IDS) or set(dota_report()) >= set(CAP_IDS)


def dota_job():
    import pandas as pd
    ann = pd.read_excel(CAP_RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
    type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
    alignment = json.loads(ALIGN.read_text())
    final_ids = set(json.loads(FINAL.read_text())['ids'])
    assert set(CAP_IDS) <= final_ids, 'a CAP-dev clip is not finalized'
    need_cap = {alignment['clips'][d]['cap_id'] for d in CAP_IDS if not (L_AG['dota'] / f'{d}.npy').exists()}
    cmds = []
    for g in CAP_GROUPS:
        lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
        ids = sorted(c for c in need_cap if lo <= type_of[c] <= hi)
        parts = sorted((CAP_RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
        assert parts, f'no parts for group {g}'
        if ids:
            cmds.append([sys.executable, '-m', 'core.tools.v6_agents', 'extract-dota', '--dev-only',
                         '--alignment', ALIGN, '--final-readout', FINAL, '--parts', *parts,
                         '--work-dir', LOCAL / 'work' / g, '--dota-clip-dir', DOTA_S1,
                         '--agent-dir', L_AG['dota'], '--track-dir', L_TR['dota'],
                         '--report', L_REP / f'extract_agents_dota_{g}.json',
                         '--cap-ids-file', write_ids(LOCAL / f'cap_ids_{g}.txt', ids),
                         '--max-batch-gb', str(constants.MMAU_STREAM_BATCH_GB), '--weights', WEIGHTS,
                         '--device', 'auto'])
    return Job('extract_dota', cmds)


if not dota_done():
    mirror = Mirror([(L_AG['dota'], D_AG['dota']), (L_TR['dota'], D_TR['dota']), (L_REP, REPORTS)])
    mirror.start()
    t0 = time.time()
    try:
        run_parallel([dota_job()], max_jobs=1)
    finally:
        mirror.stop()
    print(f'step 1 wall {(time.time() - t0) / 60:.0f} min')
    flush_drive()
rep = dota_report()
bad = {v: e['reason'] for v, e in rep.items() if e.get('reason') != 'ok'}
have = [v for v in CAP_IDS if (L_AG['dota'] / f'{v}.npy').exists()]
print(f'CAP-dev agent rows: {len(have)} / {len(CAP_IDS)}; failed {len(bad)} {sorted(bad.items())[:10]}')
assert len(have) >= 0.98 * len(CAP_IDS), 'too many CAP-dev clips failed extraction -- paste the reports to Claude'


## 2. Step 0 — go / no-go (CPU + CLIP text) → **paste `REPORTS/step0.md` to Claude**

PREREG_V6 §1 = proposal §4.3, applied mechanically: **GO** (recall@0.3 ≥ 0.80 overall, ≥ 0.70 non-ego and smallest box,
and a training-free score beats the centre prior with CI > 0), **NARROW** (train, drop strata < 0.50), **NO-GO**
(recall < 0.60: stop here).

In [ ]:
S0 = L_REP / 'step0.json'
if not S0.exists():
    run([sys.executable, '-m', 'core.tools.v6_agents', 'step0', '--agent-dir', L_AG['dota'],
         '--track-dir', L_TR['dota'], '--annotations', ANN, '--out-dir', L_REP, '--device', 'auto'])
    sync_dir(L_REP, REPORTS)
show('step0')
DECISION = json.loads(S0.read_text())['decision']
print('Step 0:', DECISION['label'])
assert DECISION['label'] != 'NO-GO', 'Step 0 NO-GO: the agent branch is not trained (PREREG_V6 §1) -- paste step0.md'


## 3. Nexar window sources: agents at `s1[::8]` (GPU, sharded)

Every window source (all positives; the corpus cuts its negatives inside them). Detection every `round(fps / 10)`-th
frame; VideoMAE clips step 3 (A2's); row count checked against A2's `Nexar_s8` bake. Resumable.

In [ ]:
from core.data.windows import load_windows

census = json.loads(CENSUS.read_text())
TRAIN_IDS = sorted({w.source for w in load_windows(DATA).values()})
assert all(census['videos'][v]['label'] == 1 for v in TRAIN_IDS), 'a window source is not a positive'
todo = [v for v in TRAIN_IDS if not (L_AG['train'] / f'{v}.npy').exists()]
print(f'Nexar sources: {len(TRAIN_IDS)}, to extract {len(todo)}')
if todo:
    ids_file = write_ids(LOCAL / 'nexar_ids.txt', todo)
    jobs = [Job(f'extract_nexar_{i}', [[sys.executable, '-m', 'core.tools.v6_agents', 'extract-nexar',
                                         '--raw', RAW, '--census', CENSUS, '--data-dir', DATA, '--clip-dir', A2_BAKE,
                                         '--agent-dir', L_AG['train'], '--track-dir', L_TR['train'],
                                         '--report', L_REP / 'extract_agents_nexar.json', '--ids-file', ids_file,
                                         '--shard', f'{i}/{NEXAR_SHARDS}', '--weights', WEIGHTS,
                                         '--device', 'auto']])
            for i in range(NEXAR_SHARDS)]
    mirror = Mirror([(L_AG['train'], D_AG['train']), (L_TR['train'], D_TR['train']), (L_REP, REPORTS)])
    mirror.start()
    t0 = time.time()
    try:
        run_parallel(jobs, max_jobs=NEXAR_SHARDS)
    finally:
        mirror.stop()
    print(f'step 3 wall {(time.time() - t0) / 60:.0f} min')
    flush_drive()
missing = [v for v in TRAIN_IDS if not (L_AG['train'] / f'{v}.npy').exists()]
assert not missing, f'{len(missing)} Nexar sources without agent rows: {missing[:5]}'
summ = [json.loads(p.read_text()) for p in L_REP.glob('extract_agents_nexar_*of*.json')]
rows = [e for s in summ for e in s.values() if e.get('reason') == 'ok']
print(f'Nexar: {len(rows)} sources, agents / row {np.mean([e["agents_mean"] for e in rows]):.2f}, '
      f'rows at K_max {np.sum([e["rows_at_k_max"] for e in rows]) / np.sum([e["rows"] for e in rows]):.3f}')


## 4. Train AP s2024 (GPU) — A2's exact recipe + `agent.enabled=true`, then J10

In [ ]:
assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
VERIFIED = 'verified.json'
RUN = ROOT / ROW / f's{SEED}'


def is_verified(d):
    rec, ck = d / VERIFIED, d / 'checkpoint_last.pt'
    if not (rec.exists() and ck.exists()):
        return False
    r = json.loads(rec.read_text())
    return r['step'] == STEPS and r['ckpt_bytes'] == ck.stat().st_size


def train_cmd(out_dir, data, a2, knn, agents):
    return [sys.executable, '-m', 'core.train',
            '--set', 'train.stage=2', '--set', f'train.seed={SEED}',
            '--set', f'train.num_epochs={EPOCHS}', '--set', 'train.amp=true',
            '--set', f'data.dataset={DATASET}',
            '--set', f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}',
            '--set', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
            '--set', 'kip.enabled=false', '--set', 'loss.dvs_anchor_mode=span',
            '--set', 'data.is_egocentric=false',
            '--set', f'v2.motion={constants.V5_ENCODER}', '--set', f'model.motion_dim={constants.V5_GRID_DIM}',
            '--set', 'agent.enabled=true',
            '--data-dir', data, '--clip-dir', a2, '--knn-cache', knn, '--agent-dir', agents,
            '--output-dir', out_dir]


if not is_verified(RUN):
    local = LOCAL / 'runs' / ROW / f's{SEED}'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    L_DATA = stage(DATA, LOCAL / 'stage' / f'data_{C}')
    L_A2 = stage(A2_BAKE, LOCAL / 'stage' / 'A2_s8')
    L_KNN = stage(KNN, LOCAL / 'stage' / f'knn_{C}.npz')

    def after():
        last = metrics_last(local / 'metrics.jsonl')
        assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, 'AP: J10 locally'
        for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
            publish(local / name, RUN / name)

    t0 = time.time()
    run_parallel([Job(f'train_{ROW}_s{SEED}', [train_cmd(local, L_DATA, L_A2, L_KNN, L_AG['train']) + resume],
                      after)], max_jobs=1)
    print(f'step 4 wall {(time.time() - t0) / 60:.0f} min')
    flush_drive()
    assert ckpt_step(RUN / 'checkpoint_last.pt') == STEPS, f'Drive copy is not step {STEPS}'
    (LOCAL / VERIFIED).write_text(json.dumps({'step': STEPS, 'ckpt_bytes': (RUN / 'checkpoint_last.pt').stat().st_size}))
    publish(LOCAL / VERIFIED, RUN / VERIFIED)
m = [json.loads(l) for l in (RUN / 'metrics.jsonl').read_text().splitlines() if l.strip()]
for key in ('agent', 'agent_alpha', 'agent_share', 'agent_top_abnormal', 'agent_top_normal', 'agent_ego_pick',
            'agent_none_pick', 'motion_share', 'mil'):
    xs = [r[key] for r in m if key in r]
    if xs:
        print(f'{key:20s} first {xs[0]:.4f}  last {xs[-1]:.4f}')


## 5. Protocol B on DoTA-CAP-dev (GPU): AP s2024 + AP candidate logits

In [ ]:
def ckpt_local(src, name):
    dst = LOCAL / 'ckpts' / name
    for f in ('metrics.jsonl', 'config.yaml', 'checkpoint_last.pt'):
        stage(src.parent / f, dst / f)
    return dst / 'checkpoint_last.pt'


PB = ROOT / 'eval' / 'dota' / CAP_DEV
LOGITS = ROOT / 'eval' / 'agent_logits' / f'{ROW}_s{SEED}.npz'
want = [r for r in (ROW,) if not (PB / r / 'protocol_b_readout.json').exists()]
jobs = []
if want or not LOGITS.exists():
    d_s1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
    d_a2 = stage(DOTA_A2, LOCAL / 'stage' / 'dota_cap_dev_A2', CAP_IDS)
    d_lab = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
    ck = {ROW: ckpt_local(RUN / 'checkpoint_last.pt', f'{ROW}_s{SEED}')}
    for r in want:
        out = LOCAL / 'pb' / r
        jobs.append(Job(f'pb_{r}', [[sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', ck[r],
                                     '--input-dir', d_a2, '--agent-dir', L_AG['dota'], '--s1-dir', d_s1,
                                     '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
                                     '--data-dir', d_lab, '--split', CAP_DEV, '--out-dir', out, '--device', 'auto']],
                        lambda out=out, r=r: publish(out, PB / r)))
    if not LOGITS.exists():
        out = LOCAL / 'logits' / LOGITS.name
        jobs.append(Job('dump', [[sys.executable, '-m', 'core.tools.v6_agents', 'dump', '--ckpt', ck[ROW],
                                  '--input-dir', d_a2, '--s1-dir', d_s1, '--agent-dir', L_AG['dota'], '--out', out]],
                        lambda out=out: publish(out, LOGITS)))
run_parallel(jobs)
if jobs:
    flush_drive()


## 6. Read-out (CPU) → **paste `REPORTS/v6_readout.md` to Claude**

Rows: AP s2024 and v5 R1 s2024 (VideoMAE only, no MQR; its existing protocol-B output). Subsets all / ego / non-ego (DoTA `ego_involve`). Paired Δ by source-video cluster bootstrap. Box-level hit of AP's top agent vs the
centre / largest / lowest priors, unconditional and given the anomalous agent was detected.

In [ ]:
L_PB = {ROW: stage(PB / ROW, LOCAL / 'read' / ROW)}
L_R1 = stage(R1_EVAL, LOCAL / 'read' / 'R1')
L_LOG = stage(LOGITS, LOCAL / 'read' / LOGITS.name)
run([sys.executable, '-m', 'core.tools.v6_agents', 'readout',
     '--row', f'{ROW}={L_PB[ROW]}', '--row', f'R1={L_R1}', '--pair', f'{ROW}=R1',
     '--annotations', ANN, '--logits', L_LOG, '--agent-dir', L_AG['dota'], '--out-dir', L_REP])
show('v6_readout')
sync_dir(L_REP, REPORTS)
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/v6_readout.md (+ step0.md) to Claude')


## 7. Post-hoc: **R1 s2024 zero-shot on the official DADA-2000 test set** (PREREG_V6 §4) → **paste `REPORTS/dada_official/dada_readout.md`**

Printed only; §3's PILOT-FAIL is unchanged and AP is **not** read here. Set = SimpleTAD's (CVPR 2025) DADA-2000
evaluation list (`DADA2K_my_split/validation.txt` in `D2K.zip`: every clip not in LOTVS's training list). Steps:
(a) list → our ids + coverage gate Z2 (≥ 0.95, every exclusion named); (b) bake A2 rows on those clips with R1's own
Nexar A2 statistics (CPU); (c) strip R1's parameter-free mean readout; (d) **gate G-S**: stripped R1 must reproduce
R1's protocol-B scores on DoTA-CAP-dev; (e) `dada_whole_eval` (whole video, every s8 row, annotation span). Resumable.
Can run in a fresh session (setup cells 1–4 first; steps 1–6 are skipped when their outputs exist).

In [ ]:
import zipfile
import urllib.request

drive.mount('/content/drive')          # step 6 unmounted Drive
V2S       = constants.VIDEOMAE_ENCODER_S
T2NAME    = constants.DADA_ORIGIN_DATASET                          # DADA2000_orig (original release)
D_CLIP    = constants.CLIP_CACHE_DIR / T2NAME                      # CLIP s8, every source
D_VIDEO   = next((c for c in (constants.VIDEO_CACHE_DIR / V2S / f'{T2NAME}_s8_squash',
                              V2_CACHE / 'video' / V2S / f'{T2NAME}_s8_squash') if c.is_dir()), None)
D_ANNOT   = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
D_CENSUS  = constants.DATA_ROOT / T2NAME / 'counts' / 'resolved_census.json'
R1_CKPT   = Path(DRIVE_V5) / 'outputs' / 'v5' / 'v5_stage1' / 'R1' / f's{SEED}' / 'checkpoint_last.pt'
DD        = constants.V6_DADA_DIRNAME
OUT_DADA  = ROOT / 'eval' / DD / 'R1'                                # Thesis-V6/outputs/v6/eval/dada_official/R1
REP_DADA  = REPORTS / DD
BAKE_DADA = CACHE6 / 'v2_bake' / 'A2_S' / f'{T2NAME}_{DD}_nexar'    # Thesis-V6/cache: R1's Nexar A2 statistics
L_DD      = LOCAL / DD
for name, path in (('DADA CLIP s8', D_CLIP), ('DADA V2-S s8', D_VIDEO), ('annotation', D_ANNOT),
                   ('census', D_CENSUS), ('R1 ckpt', R1_CKPT), ('R1 DoTA-CAP-dev eval', R1_EVAL / 'clip_scores.npz'),
                   ('A2 Nexar stats', A2_BAKE / 'v2_input_stats.npz')):
    assert path is not None and path.exists(), f'missing {name}: {path}'
    print(f'  {name:22s} {path}')
for p in (REP_DADA, L_DD / 'rep'):
    p.mkdir(parents=True, exist_ok=True)
sync_dir(REP_DADA, L_DD / 'rep', restore=True)

# (a) official list -> ids, coverage gate Z2 (the tool exits on < 0.95)
IDS_JSON = L_DD / 'rep' / 'dada_official_ids.json'
if not IDS_JSON.exists():
    zpath = LOCAL / 'D2K.zip'
    if not zpath.exists():
        urllib.request.urlretrieve(constants.V6_DADA_OFFICIAL_URL, zpath)   # nosec B310 - fixed https URL
    with zipfile.ZipFile(zpath) as z:
        member = next(n for n in z.namelist() if n.endswith(constants.V6_DADA_OFFICIAL_LIST))
        (L_DD / 'validation.txt').write_bytes(z.read(member))
    run([sys.executable, '-m', 'core.tools.v6_r1_dada', 'ids', '--official', L_DD / 'validation.txt',
         '--annotation', D_ANNOT, '--census', D_CENSUS, '--clip-dir', D_CLIP, '--video-dir', D_VIDEO,
         '--out-dir', L_DD / 'rep'])
    sync_dir(L_DD / 'rep', REP_DADA)
ids_rep = json.loads(IDS_JSON.read_text())
DADA_IDS = [l for l in (L_DD / 'rep' / 'dada_official_ids.txt').read_text().split() if l]
print(f"official list {ids_rep['official']} clips -> read {ids_rep['kept']} (coverage {ids_rep['coverage']:.3f}); "
      f"dropped { {k: len(v) for k, v in ids_rep['dropped'].items()} }")
assert ids_rep['passes'] and len(DADA_IDS) == ids_rep['kept']

# (b) bake A2 rows on those sources with R1's Nexar A2 statistics (CPU)
man = BAKE_DADA / constants.V2_INPUT_MANIFEST_FILENAME
if not (man.exists() and json.loads(man.read_text())['baked_ids'] == len(DADA_IDS)):
    ids_file = write_ids(L_DD / 'ids.txt', DADA_IDS)
    l_x = stage(D_CLIP, L_DD / 'stage' / 'clip', DADA_IDS)
    l_u = stage(D_VIDEO, L_DD / 'stage' / 'v2s', DADA_IDS)
    l_stats = stage(A2_BAKE, LOCAL / 'stage' / 'A2_s8')
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', l_stats, '--clip-dir', l_x,
         '--video-dir', l_u, '--ids-file', ids_file, '--stride', 1, '--out-dir', L_DD / 'bake'], log=LOGS / 'dada_bake.log')
    publish(L_DD / 'bake', BAKE_DADA)
m = json.loads(man.read_text())
nexar_m = json.loads((A2_BAKE / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
assert (m['crn'], m['motion']) == (constants.V2_OFF, V2S) and m['train_ids_sha1'] == nexar_m['train_ids_sha1'], m
L_BAKE = stage(BAKE_DADA, L_DD / 'stage' / 'bake', DADA_IDS)

# (c) strip R1's mean readout (same weights, v5.readout=none, reads the baked u)
STRIP = L_DD / 'R1_stripped' / f's{SEED}'
if not (STRIP / 'checkpoint_last.pt').exists():
    r1_local = stage(R1_CKPT.parent / 'metrics.jsonl', L_DD / 'R1' / 'metrics.jsonl').parent
    stage(R1_CKPT.parent / 'config.yaml', r1_local / 'config.yaml')
    stage(R1_CKPT, r1_local / 'checkpoint_last.pt')
    run([sys.executable, '-m', 'core.tools.v6_r1_dada', 'strip', '--ckpt', r1_local / 'checkpoint_last.pt',
         '--out-dir', STRIP])

# (d) gate G-S on DoTA-CAP-dev: stripped R1 on A2's baked rows == R1's own protocol-B scores
GS = L_DD / 'rep' / 'g_s.json'
if not GS.exists():
    d_s1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
    d_a2 = stage(DOTA_A2, LOCAL / 'stage' / 'dota_cap_dev_A2', CAP_IDS)
    d_lab = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', STRIP / 'checkpoint_last.pt',
         '--input-dir', d_a2, '--s1-dir', d_s1, '--metadata', DOTA / 'metadata_val.json',
         '--split-file', DOTA / 'val_split.txt', '--data-dir', d_lab, '--split', CAP_DEV,
         '--out-dir', L_DD / 'gs_pb', '--device', 'auto'], log=LOGS / 'dada_gs.log')
    run([sys.executable, '-m', 'core.tools.v6_r1_dada', 'gate', '--reference', R1_EVAL / 'clip_scores.npz',
         '--candidate', L_DD / 'gs_pb' / 'clip_scores.npz', '--out', GS])
    sync_dir(L_DD / 'rep', REP_DADA)
g = json.loads(GS.read_text())
print(f"G-S {'PASS' if g['passes'] else 'FAIL'}: macro R1 {g['macro_reference']:.4f} vs stripped {g['macro_candidate']:.4f}, "
      f"max |d score| {g['max_abs_score_delta']:.2e}")
assert g['passes'], 'G-S FAIL: no DADA number is read -- paste g_s.json to Claude'

# (e) R1 on the official DADA-2000 list: whole videos, every s8 row
if not (OUT_DADA / 'dada_readout.json').exists():
    l_data = stage(DATA, LOCAL / 'stage' / f'data_{C}')
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', STRIP / 'checkpoint_last.pt',
         '--ids-file', L_DD / 'rep' / 'dada_official_ids.txt',
         '--set-name', f"DADA-2000 official test list (SimpleTAD validation.txt, {len(DADA_IDS)} of {ids_rep['official']})",
         '--annotation', D_ANNOT, '--census', D_CENSUS, '--input-dir', L_BAKE, '--data-dir', l_data,
         '--device', 'auto', '--out-dir', L_DD / 'eval'], log=LOGS / 'dada_eval.log')
    publish(L_DD / 'eval', OUT_DADA)
    shutil.copy2(OUT_DADA / 'dada_readout.md', L_DD / 'rep' / 'dada_readout.md')
    sync_dir(L_DD / 'rep', REP_DADA)
print((OUT_DADA / 'dada_readout.md').read_text())
drive.flush_and_unmount()
print('Drive flushed -- paste REPORTS/dada_official/{dada_readout.md, g_s.json, dada_official_ids.json} to Claude')